# Khung Code Ôn Thi TX2 – Khai Phá Dữ Liệu (Bản Nâng Cấp Tổng Quát v2)

Tổng hợp đầy đủ **Code chay (thuần thuật toán)** + **Code thư viện (gọi hàm)** cho mọi thuật toán trong đề cương TX2, đã nâng cấp để chạy được trên **mọi file `.csv`** (2 lớp hay nhiều lớp, có chữ, có `NaN`, cột `class` ở bất kỳ vị trí nào) nhưng vẫn giữ **ngắn gọn – dễ thuộc – dễ code nhanh**.

**Checklist 30 giây khi vào phòng thi:**
1. Đổi tên file CSV và vị trí cột nhãn `target_col` ở **Cell 1**.
2. Đổi đoạn chuẩn hóa `[a, b]` và giá trị mẫu mới `X_new_raw`.
3. Chạy đúng cell thuật toán đề yêu cầu (thay `K` nếu là phân cụm, `min_sup/min_conf` nếu là luật kết hợp).


In [1]:
# !pip install numpy pandas scikit-learn mlxtend
# !pip install matplotlib seaborn scipy


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 1: Tiền xử lý tổng quát (Đọc CSV, xử lý NaN/chữ, Chuẩn hóa [a,b], Chia Train/Test)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# 1. Đọc dữ liệu & làm sạch cơ bản
df = pd.read_csv('iris.csv')          # Đổi tên file theo đề bài
df = df.dropna().reset_index(drop=True)  # Bỏ các dòng thiếu dữ liệu (NaN) để tránh lỗi
# df = df.drop(columns=['CustomerID'])   # Mở dòng này nếu file có cột ID thừa không dùng

# 2. Tách đặc trưng (X) và nhãn (y) - linh hoạt mọi vị trí cột class
target_col = df.columns[-1]           # Mặc định cột cuối (đổi thành df.columns[0] nếu ở đầu)
X_df = df.drop(columns=[target_col]).copy()
y_raw = df[target_col].values

# Số hóa cột class (chữ -> số 0, 1, 2...)
le = LabelEncoder()
y = le.fit_transform(y_raw)

# 3. Nhập mẫu mới X_new theo đúng thứ tự các cột trong X_df
X_new_df = pd.DataFrame([[6, 3, 0.7, 0.9]], columns=X_df.columns)  # Đổi giá trị theo đề

# Tự động số hóa các cột đặc trưng dạng CHỮ (nếu có, vd: cột Gender)
for col in X_df.columns:
    if X_df[col].dtype == 'object':
        enc = LabelEncoder()
        X_df[col] = enc.fit_transform(X_df[col])
        X_new_df[col] = enc.transform(X_new_df[col])

X_raw = X_df.values.astype(float)
X_new_raw = X_new_df.values.astype(float)

# 4. Chuẩn hóa dữ liệu về đoạn [a, b]
a, b = 0, 1                           # Đổi đoạn [a, b] theo đề bài (vd: [0, 10], [0, 20])

# --- Cách 1: Gọi hàm thư viện ---
scaler = MinMaxScaler(feature_range=(a, b))
X = scaler.fit_transform(X_raw)       # Học min/max từ X_raw rồi chuẩn hóa
X_new = scaler.transform(X_new_raw)   # BẮT BUỘC dùng .transform() cho mẫu mới

# --- Cách 2: Code chay (bỏ comment nếu đề yêu cầu chuẩn hóa thuần) ---
# X_min, X_max = X_raw.min(axis=0), X_raw.max(axis=0)
# X = a + (X_raw - X_min) * (b - a) / (X_max - X_min + 1e-9)
# X_new = a + (X_new_raw - X_min) * (b - a) / (X_max - X_min + 1e-9)

# 5. Chia tập Train / Test (80% - 20%)
# --- Cách 1: Gọi hàm thư viện ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Cách 2: Code chay (bỏ comment nếu đề yêu cầu chia thuần) ---
# np.random.seed(42)
# idx = np.random.permutation(len(X))
# split = int(0.8 * len(X))
# X_train, X_test = X[idx[:split]], X[idx[split:]]
# y_train, y_test = y[idx[:split]], y[idx[split:]]

print(f"Xong tiền xử lý! Số mẫu: {len(X)}, Số cột: {X.shape[1]}, Các lớp: {le.classes_}")


## Cell 2: Phân lớp Naive Bayes (Code chay & Thư viện – Hỗ trợ mọi số lớp)


In [ ]:
# ================= PHẦN 1: CODE CHAY (GAUSSIAN NAIVE BAYES) =================
def gaussian_pdf(x, mean, var):
    eps = 1e-9  # Tránh chia cho 0 khi phương sai = 0
    return (1.0 / np.sqrt(2.0 * np.pi * (var + eps))) * np.exp(-((x - mean) ** 2) / (2 * (var + eps)))

classes = np.unique(y)
probs = {}

for c in classes:
    X_c = X[y == c]                         # Lọc các mẫu thuộc lớp c
    prior = len(X_c) / len(X)               # Xác suất tiên nghiệm P(C)
    mean, var = X_c.mean(axis=0), X_c.var(axis=0)
    likelihood = np.prod(gaussian_pdf(X_new[0], mean, var))  # Tích P(x_i | C)
    probs[c] = prior * likelihood

# Chuẩn hóa để tổng xác suất các lớp = 1
total = sum(probs.values()) + 1e-15
for c in classes:
    probs[c] /= total
    print(f"P(X thuộc {le.inverse_transform([c])[0]}) = {probs[c]:.4f}")

pred_c = max(probs, key=probs.get)
print("=> Kết luận (Code chay): X thuộc lớp", le.inverse_transform([pred_c])[0])


# ================= PHẦN 2: CODE THƯ VIỆN (KIỂM NGHIỆM) =================
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X, y)  # Đổi thành (X_train, y_train) nếu đề yêu cầu đánh giá trên tập test
print("Xác suất (Thư viện):", np.round(nb.predict_proba(X_new)[0], 4))
print("=> Kết luận (Thư viện): X thuộc lớp", le.inverse_transform(nb.predict(X_new))[0])


## Cell 3: Phân cụm K-Means (Code chay chống cụm rỗng & Thư viện)


In [ ]:
# ================= PHẦN 1: CODE CHAY K-MEANS =================
K = 2  # Đổi số cụm theo đề bài
np.random.seed(42)

# Bước 1: Chọn ngẫu nhiên K điểm làm tâm ban đầu
centroids = X[np.random.choice(len(X), K, replace=False)]

for _ in range(100):
    # Bước 2: Tính khoảng cách Euclid từ mỗi điểm đến K tâm và gán cụm gần nhất
    distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
    labels = np.argmin(distances, axis=1)
    
    # Bước 3: Cập nhật tâm mới (giữ nguyên tâm cũ nếu chẳng may cụm bị rỗng)
    new_centroids = np.array([
        X[labels == k].mean(axis=0) if np.any(labels == k) else centroids[k]
        for k in range(K)
    ])
    if np.allclose(centroids, new_centroids):
        break
    centroids = new_centroids

for k in range(K):
    print(f"[Chay] Cụm {k}: {np.sum(labels == k)} mẫu | Tâm: {np.round(centroids[k], 3)}")

print("=> [Chay] X_new thuộc cụm:", np.argmin(np.linalg.norm(centroids - X_new[0], axis=1)))


# ================= PHẦN 2: CODE THƯ VIỆN K-MEANS =================
from sklearn.cluster import KMeans

km = KMeans(n_clusters=K, random_state=42, n_init=10)
labels_lib = km.fit_predict(X)

for k in range(K):
    print(f"[Thư viện] Cụm {k}: {np.sum(labels_lib == k)} mẫu | Tâm: {np.round(km.cluster_centers_[k], 3)}")

print("=> [Thư viện] X_new thuộc cụm:", km.predict(X_new)[0])


## Cell 4: Phân cụm phân cấp – Thuật toán gộp Agglomerative (Code chay & Thư viện)


In [ ]:
# ================= PHẦN 1: CODE CHAY THUẬT TOÁN GỘP =================
K = 3  # Đổi số cụm mục tiêu (vd: Đề 1 K=3, Đề 2 K=4)

# Bước 1: Ban đầu mỗi điểm là 1 cụm riêng
clusters = [[i] for i in range(len(X))]
centers = [X[i] for i in range(len(X))]

# Bước 2: Gộp dần 2 cụm có tâm gần nhau nhất cho đến khi còn đúng K cụm
while len(clusters) > K:
    min_dist, pair = float('inf'), (0, 1)
    for i in range(len(centers)):
        for j in range(i + 1, len(centers)):
            d = np.linalg.norm(centers[i] - centers[j])
            if d < min_dist:
                min_dist, pair = d, (i, j)
    
    i, j = pair
    clusters[i].extend(clusters[j])
    centers[i] = X[clusters[i]].mean(axis=0)
    clusters.pop(j)
    centers.pop(j)

for idx, c in enumerate(clusters):
    print(f"[Chay] Cụm {idx}: {len(c)} mẫu")

print("=> [Chay] X_new thuộc cụm:", np.argmin([np.linalg.norm(c - X_new[0]) for c in centers]))


# ================= PHẦN 2: CODE THƯ VIỆN =================
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(n_clusters=K, linkage='ward')
labels_agg = agg.fit_predict(X)

centers_lib = []
for k in range(K):
    print(f"[Thư viện] Cụm {k}: {np.sum(labels_agg == k)} mẫu")
    centers_lib.append(X[labels_agg == k].mean(axis=0))

print("=> [Thư viện] X_new thuộc cụm:", np.argmin([np.linalg.norm(c - X_new[0]) for c in centers_lib]))


## Cell 5: Cây quyết định ID3 / CART (Code thư viện & Code chay tự xử lý cả dữ liệu số/chữ)


In [ ]:
# ================= PHẦN 1: CODE THƯ VIỆN =================
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# criterion='entropy' (ID3) hoặc 'gini' (CART)
clf = DecisionTreeClassifier(criterion='entropy', random_state=42)
clf.fit(X_train, y_train)
print("[Thư viện] Accuracy:", round(accuracy_score(y_test, clf.predict(X_test)), 4))
print("[Thư viện] Dự đoán X_new:", le.inverse_transform(clf.predict(X_new))[0])


# ================= PHẦN 2: CODE CHAY ID3 (CHẠY ĐƯỢC CẢ DỮ LIỆU SỐ & CHỮ) =================
def entropy(y_col):
    _, counts = np.unique(y_col, return_counts=True)
    p = counts / len(y_col)
    return -np.sum(p * np.log2(p))

def info_gain(df_in, feature, target):
    total_ent = entropy(df_in[target])
    vals, counts = np.unique(df_in[feature], return_counts=True)
    weighted_ent = sum((counts[i] / len(df_in)) * entropy(df_in[df_in[feature] == vals[i]][target])
                       for i in range(len(vals)))
    return total_ent - weighted_ent

def build_tree(df_in, features, target):
    if len(np.unique(df_in[target])) == 1:      # Tất cả cùng 1 nhãn -> Nút lá
        return df_in[target].iloc[0]
    if len(features) == 0:                      # Hết thuộc tính -> lấy nhãn phổ biến nhất
        return df_in[target].mode()[0]
    
    best_feat = features[np.argmax([info_gain(df_in, f, target) for f in features])]
    tree = {best_feat: {}}
    rem_feats = [f for f in features if f != best_feat]
    
    for val in np.unique(df_in[best_feat]):
        tree[best_feat][val] = build_tree(df_in[df_in[best_feat] == val], rem_feats, target)
    return tree

def predict_one(tree, sample, default_label):
    if not isinstance(tree, dict):
        return tree
    feat = next(iter(tree))
    val = sample[feat]
    return predict_one(tree[feat][val], sample, default_label) if val in tree[feat] else default_label

# Chuẩn bị bảng dữ liệu cho ID3 chay (tự động rời rạc hóa nếu cột là số liên tục)
df_id3 = df.copy()
sample_id3 = X_new_df.iloc[0].to_dict()
feat_cols = [c for c in df_id3.columns if c != target_col]

for col in feat_cols:
    if pd.api.types.is_numeric_dtype(df_id3[col]) and df_id3[col].nunique() > 5:
        # Chia cột số thành 3 khoảng (low/med/high) và lưu mốc 'bins' để cắt đúng X_new
        df_id3[col], bins = pd.cut(df_id3[col], bins=3, labels=['low', 'med', 'high'], retbins=True)
        bins[0], bins[-1] = -np.inf, np.inf  # Mở rộng 2 đầu để X_new ngoài biên không bị NaN
        sample_id3[col] = pd.cut([sample_id3[col]], bins=bins, labels=['low', 'med', 'high'])[0]

tree = build_tree(df_id3, feat_cols, target_col)
default_lbl = df_id3[target_col].mode()[0]
y_pred_id3 = [predict_one(tree, row, default_lbl) for _, row in df_id3.iterrows()]

print(f"[Chay ID3] Accuracy (trên tập gốc): {np.mean(y_pred_id3 == df_id3[target_col].values):.4f}")
print("[Chay ID3] Dự đoán X_new:", predict_one(tree, sample_id3, default_lbl))


## Cell 6: Mạng Nơ-ron 1 lớp (Perceptron) & Nhiều lớp (MLP) – Hỗ trợ mọi số lớp


In [ ]:
# ================= PHẦN 1: CODE THƯ VIỆN (1 LỚP & NHIỀU LỚP) =================
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier

# 1. Perceptron 1 lớp (Thư viện)
ppn = Perceptron(max_iter=200, eta0=0.01, random_state=42)
ppn.fit(X_train, y_train)
print("[Thư viện - Perceptron] Accuracy:", round(ppn.score(X_test, y_test), 4),
      "| X_new:", le.inverse_transform(ppn.predict(X_new))[0])

# 2. MLP nhiều lớp (Thư viện)
mlp = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
print("[Thư viện - MLP] Accuracy:", round(mlp.score(X_test, y_test), 4),
      "| X_new:", le.inverse_transform(mlp.predict(X_new))[0])


# ================= PHẦN 2: CODE CHAY PERCEPTRON 1 LỚP (ĐA LỚP) =================
n_features = X_train.shape[1]
n_classes = len(np.unique(y))

W = np.zeros((n_features, n_classes))   # Mỗi cột là trọng số của 1 lớp
b = np.zeros(n_classes)
lr = 0.01

for _ in range(100):
    for xi, yi in zip(X_train, y_train):
        y_hat = np.argmax(xi @ W + b)   # Chọn lớp có điểm số cao nhất
        if y_hat != yi:                 # Nếu đoán sai: cộng vào lớp đúng, trừ ở lớp sai
            W[:, yi] += lr * xi;    b[yi] += lr
            W[:, y_hat] -= lr * xi; b[y_hat] -= lr

pred_test_p = np.argmax(X_test @ W + b, axis=1)
pred_new_p = np.argmax(X_new[0] @ W + b)
print("[Chay - Perceptron] Accuracy:", round(np.mean(pred_test_p == y_test), 4),
      "| X_new:", le.inverse_transform([pred_new_p])[0])


# ================= PHẦN 3: CODE CHAY MLP NHIỀU LỚP (1 LỚP ẨN + SOFTMAX ĐA LỚP) =================
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))  # Trừ max để tránh tràn số
    return e / e.sum(axis=1, keepdims=True)

np.random.seed(42)
n_hidden = 10
W1 = np.random.randn(n_features, n_hidden) * 0.1;  b1 = np.zeros((1, n_hidden))
W2 = np.random.randn(n_hidden, n_classes) * 0.1;   b2 = np.zeros((1, n_classes))

Y_oh = np.eye(n_classes)[y_train]       # Mã hóa One-Hot cho nhãn train (vd: 2 -> [0, 0, 1])
lr = 0.1

for _ in range(1000):
    # Lan truyền tiến (Forward)
    A1 = sigmoid(X_train @ W1 + b1)
    A2 = softmax(A1 @ W2 + b2)
    
    # Lan truyền ngược (Backpropagation)
    dZ2 = A2 - Y_oh
    dW2 = (A1.T @ dZ2) / len(X_train);  db2 = np.mean(dZ2, axis=0, keepdims=True)
    
    dZ1 = (dZ2 @ W2.T) * A1 * (1 - A1)
    dW1 = (X_train.T @ dZ1) / len(X_train);  db1 = np.mean(dZ1, axis=0, keepdims=True)
    
    # Cập nhật trọng số
    W2 -= lr * dW2;  b2 -= lr * db2
    W1 -= lr * dW1;  b1 -= lr * db1

pred_test_m = np.argmax(softmax(sigmoid(X_test @ W1 + b1) @ W2 + b2), axis=1)
pred_new_m = np.argmax(softmax(sigmoid(X_new @ W1 + b1) @ W2 + b2)[0])
print("[Chay - MLP] Accuracy:", round(np.mean(pred_test_m == y_test), 4),
      "| X_new:", le.inverse_transform([pred_new_m])[0])


## Cell 7: Luật kết hợp Apriori & FP-Growth (Code chay & Thư viện)


In [ ]:
# Đọc dữ liệu giao dịch (mỗi dòng là 1 giỏ hàng, có thể chứa NaN)
# df_basket = pd.read_csv('Market_Basket_Optimisation.csv', header=None)

# Ví dụ dữ liệu mẫu nhỏ để chạy thử ngay (thay bằng df_basket.values khi làm bài thật)
sample_baskets = [
    ['milk', 'bread', 'eggs'],
    ['milk', 'bread'],
    ['milk', 'eggs'],
    ['bread', 'eggs'],
    ['milk', 'bread', 'eggs']
]
# Khi dùng file CSV thật, mở dòng dưới:
# transactions = [set(str(item) for item in row if pd.notna(item)) for row in df_basket.values]
transactions = [set(row) for row in sample_baskets]


# ================= PHẦN 1: CODE CHAY APRIORI =================
from itertools import combinations

N = len(transactions)
min_sup, min_conf = 0.4, 0.6

def get_sup(itemset):
    return sum(1 for t in transactions if itemset.issubset(t)) / N

# Bước 1: Tìm tập 1 phần tử phổ biến (L1)
items = sorted(set().union(*transactions))
L = [{frozenset([i]): get_sup({i}) for i in items if get_sup({i}) >= min_sup}]

# Bước 2: Ghép dần tìm L2, L3...
k = 2
while L[-1]:
    prev = list(L[-1].keys())
    cands = {a | b for a in prev for b in prev if len(a | b) == k}
    Lk = {c: get_sup(c) for c in cands if get_sup(c) >= min_sup}
    if not Lk:
        break
    L.append(Lk)
    k += 1

# Bước 3: Sinh luật kết hợp A -> B
print("--- Luật kết hợp (Code chay Apriori) ---")
for Lk in L[1:]:
    for itemset, sup_AB in Lk.items():
        for i in range(1, len(itemset)):
            for A in combinations(itemset, i):
                A = frozenset(A)
                B = itemset - A
                conf = sup_AB / get_sup(A)
                lift = conf / get_sup(B)
                if conf >= min_conf:
                    print(f"{set(A)} -> {set(B)} | sup={sup_AB:.2f}, conf={conf:.2f}, lift={lift:.2f}")


# ================= PHẦN 2: CODE THƯ VIỆN (APRIORI & FP-GROWTH) =================
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

te = TransactionEncoder()
df_enc = pd.DataFrame(te.fit_transform([list(t) for t in transactions]), columns=te.columns_)

# Chọn fpgrowth hoặc apriori tùy đề bài
freq_items = fpgrowth(df_enc, min_support=min_sup, use_colnames=True)
# freq_items = apriori(df_enc, min_support=min_sup, use_colnames=True)

rules = association_rules(freq_items, metric="confidence", min_threshold=min_conf)
print("\n--- Luật kết hợp (Thư viện) ---")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())
